# 01 · Revisión de datos: dataset de Kaggle y cobertura del cruce con el historial

Proyecto «Playlist perfecta para viajes en grupo» · NPD Entertainment · CEBANC · Ander Ameztoy

Este notebook hace tres cosas:

1. Carga el **Spotify Tracks Dataset** (maharshipandya, Kaggle) desde `data/entrada/kaggle/spotify_tracks.csv`.
2. Revisa su calidad: tamaño, nulos, duplicados, rangos y tipos.
3. Mide qué parte del historial extendido de Spotify tiene características de audio en Kaggle, cruzando por `track_id` y después por nombre de canción + artista.

**Privacidad.** Del historial solo se leen los campos de la lista `CAMPOS_HISTORIAL`. Los campos personales (`ip_addr`, `conn_country`, etc.) se descartan al leer cada registro, así que nunca llegan a un DataFrame ni a una salida. El notebook solo muestra cifras agregadas del historial, nunca filas.

Uso de IA (nivel VERDE): notebook escrito con ayuda de Claude y revisado por Ander.

## 0. Configuración

In [1]:
from pathlib import Path
import glob
import json
import re
import unicodedata
from datetime import date

import pandas as pd

# ¿Se está ejecutando en Google Colab? Entonces los datos se leen de Google Drive.
try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount("/content/drive")
    # Carpeta de Drive donde se ha descomprimido datos_colab.zip (debe contener data/entrada/...)
    RAIZ = Path("/content/drive/MyDrive/proyecto-Ebaluacion1")
else:
    # Raíz del repositorio (funciona tanto si se abre desde notebooks/ como desde la raíz)
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RUTA_KAGGLE = RAIZ / "data" / "entrada" / "kaggle" / "spotify_tracks.csv"
assert RUTA_KAGGLE.exists(), f"No encuentro {RUTA_KAGGLE}. Revisa la carpeta RAIZ."
# Dataset de respaldo (Spotify 1 Million Tracks, amitanshjoshi): fichero spotify_data.csv renombrado
RUTA_RESPALDO = RAIZ / "data" / "entrada" / "kaggle" / "spotify_1m_tracks.csv"
URL_RESPALDO = "https://www.kaggle.com/datasets/amitanshjoshi/spotify-1million-tracks"

# Una carpeta por persona con sus Streaming_History_Audio_*.json (o endsong_*.json).
# Se buscan también en subcarpetas.
# Historiales originales: data/raw/<persona>/ (entrada de pipeline.sh); en Drive y en instalaciones
# antiguas están en data/entrada/historial/<persona>/
CARPETA_HISTORIAL = RAIZ / "data" / "raw"
if not (CARPETA_HISTORIAL / "ander").exists():
    CARPETA_HISTORIAL = RAIZ / "data" / "entrada" / "historial"
HISTORIALES = {
    "ander":  CARPETA_HISTORIAL / "ander",
    "daniel": CARPETA_HISTORIAL / "daniel",
}

# Datos de la ficha (van a la sección 8.2 y a las referencias del informe)
URL_KAGGLE = "https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset"
FECHA_DESCARGA = "2026-11-19"   # <- poner la fecha real de descarga

# Cifras de referencia del historial de Ander (sección 8.1 del informe)
REFERENCIA_ANDER = {"reproducciones": 263_001, "con_uri": 262_830}

UMBRAL_RESPALDO = 0.40   # regla de decisión: < 40 % de reproducciones -> dataset de respaldo

# Únicos campos del historial que se leen. Todo lo demás (ip_addr, conn_country...) se descarta.
CAMPOS_HISTORIAL = [
    "ts", "ms_played", "spotify_track_uri",
    "master_metadata_track_name", "master_metadata_album_artist_name",
]

pd.set_option("display.max_columns", 30)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| raíz:", RAIZ)
print("CSV de Kaggle:", RUTA_KAGGLE, "| existe:", RUTA_KAGGLE.exists())

Mounted at /content/drive
Entorno: Google Colab | raíz: /content/drive/MyDrive/proyecto-Ebaluacion1
CSV de Kaggle: /content/drive/MyDrive/proyecto-Ebaluacion1/data/entrada/kaggle/spotify_tracks.csv | existe: True


## 1. Dataset de Kaggle: carga y tamaño real

In [2]:
kaggle_raw = pd.read_csv(RUTA_KAGGLE)

tam_mb = RUTA_KAGGLE.stat().st_size / 1024**2
print(f"URL de la ficha:   {URL_KAGGLE}")
print(f"Fecha de descarga: {FECHA_DESCARGA}")
print(f"Filas:    {len(kaggle_raw):,}".replace(",", "."))
print(f"Columnas: {kaggle_raw.shape[1]}")
print(f"Tamaño en disco: {tam_mb:.1f} MB | en memoria: {kaggle_raw.memory_usage(deep=True).sum()/1024**2:.1f} MB")

columnas_indice = [c for c in kaggle_raw.columns if c.startswith("Unnamed")]
if columnas_indice:
    col = columnas_indice[0]
    es_indice = (kaggle_raw[col] == pd.RangeIndex(len(kaggle_raw))).all()
    print(f"Columna de índice sobrante: {columnas_indice} (coincide con 0..n-1: {es_indice})")
else:
    print("No hay columna de índice sobrante")

print("\nColumnas:", list(kaggle_raw.columns))
kaggle_raw.head()

URL de la ficha:   https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
Fecha de descarga: 2026-11-19
Filas:    114.000
Columnas: 21
Tamaño en disco: 19.2 MB | en memoria: 49.7 MB
Columna de índice sobrante: ['Unnamed: 0'] (coincide con 0..n-1: True)

Columnas: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
# A partir de aquí se trabaja sin la columna de índice sobrante
kaggle = kaggle_raw.drop(columns=columnas_indice)
print(f"Columnas útiles: {kaggle.shape[1]}")

Columnas útiles: 20


## 2. Revisión de calidad

### 2.1 Nulos por columna

In [4]:
nulos = kaggle.isna().sum().rename("nulos").to_frame()
nulos["%"] = (nulos["nulos"] / len(kaggle) * 100).round(3)
print(f"Total de celdas nulas: {int(nulos['nulos'].sum())} | filas con algún nulo: {int(kaggle.isna().any(axis=1).sum())}")
nulos[nulos["nulos"] > 0] if (nulos["nulos"] > 0).any() else print("Sin nulos")

Total de celdas nulas: 3 | filas con algún nulo: 1


,nulos,%
artists,1,0.001
album_name,1,0.001
track_name,1,0.001


In [5]:
# Filas con nulos (son canciones del catálogo, no datos personales)
kaggle[kaggle.isna().any(axis=1)]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


### 2.2 Duplicados

Un mismo `track_id` puede aparecer varias veces, normalmente porque la canción está clasificada en varios géneros (`track_genre`). Es el problema 10 del plan de ensuciado.

In [6]:
n_ids = kaggle["track_id"].nunique()
filas_dup = int(kaggle.duplicated(subset="track_id").sum())          # filas sobrantes
ids_repetidos = int((kaggle["track_id"].value_counts() > 1).sum())   # ids que salen más de una vez

generos_por_id = kaggle.groupby("track_id")["track_genre"].nunique()
ids_varios_generos = int((generos_por_id > 1).sum())

filas_identicas = int(kaggle.duplicated().sum())
cols_sin_genero = [c for c in kaggle.columns if c != "track_genre"]
dup_solo_genero = int(kaggle.duplicated(subset=cols_sin_genero).sum())

resumen_dup = pd.Series({
    "track_id distintos": n_ids,
    "filas sobrantes por track_id repetido": filas_dup,
    "track_id que se repiten": ids_repetidos,
    "  ...de ellos con distinto track_genre": ids_varios_generos,
    "filas idénticas en todas las columnas": filas_identicas,
    "filas que solo cambian en track_genre": dup_solo_genero,
}, name="valor")
resumen_dup.to_frame()

,valor
track_id distintos,89741
filas sobrantes por track_id repetido,24259
track_id que se repiten,16641
...de ellos con distinto track_genre,16299
filas idénticas en todas las columnas,450
filas que solo cambian en track_genre,23539


In [7]:
# ¿Los duplicados cambian también en algo que no sea el género? (p. ej. popularity)
otras = [c for c in kaggle.columns if c not in ("track_id", "track_genre")]
cambia = kaggle.groupby("track_id")[otras].nunique(dropna=False)
cambia = (cambia > 1).sum()
print("track_id repetidos con valores distintos en cada columna (además del género):")
cambia[cambia > 0] if (cambia > 0).any() else print("ninguna: solo cambia track_genre")

track_id repetidos con valores distintos en cada columna (además del género):


,0
popularity,720


### 2.3 Rangos: valores imposibles o sospechosos

In [8]:
CARACT_0_1 = ["danceability", "energy", "speechiness", "acousticness",
              "instrumentalness", "liveness", "valence"]

checks = {
    "tempo = 0":                    kaggle["tempo"] == 0,
    "duration_ms = 0":              kaggle["duration_ms"] == 0,
    "popularity fuera de 0-100":    ~kaggle["popularity"].between(0, 100),
    "time_signature = 0":           kaggle["time_signature"] == 0,
    "loudness > 0 dB":              kaggle["loudness"] > 0,
    "key fuera de -1..11":          ~kaggle["key"].between(-1, 11),
    "mode fuera de {0,1}":          ~kaggle["mode"].isin([0, 1]),
}
for c in CARACT_0_1:
    checks[f"{c} fuera de 0-1"] = ~kaggle[c].between(0, 1)

rangos = pd.DataFrame({
    "filas": {k: int(v.sum()) for k, v in checks.items()},
    "track_id distintos": {k: int(kaggle.loc[v, "track_id"].nunique()) for k, v in checks.items()},
})
rangos

,filas,track_id distintos
tempo = 0,157,157
duration_ms = 0,1,1
popularity fuera de 0-100,0,0
time_signature = 0,163,162
loudness > 0 dB,90,68
key fuera de -1..11,0,0
"mode fuera de {0,1}",0,0
danceability fuera de 0-1,0,0
energy fuera de 0-1,0,0
speechiness fuera de 0-1,0,0


In [9]:
# Resumen numérico para ver mínimos y máximos de un vistazo
kaggle.describe().T[["min", "max", "mean"]].round(3)

,min,max,mean
popularity,0.000,100.000,33.239
duration_ms,0.000,5237295.000,228029.153
danceability,0.000,0.985,0.567
energy,0.000,1.000,0.641
key,0.000,11.000,5.309
loudness,-49.531,4.532,-8.259
mode,0.000,1.000,0.638
speechiness,0.000,0.965,0.085
acousticness,0.000,0.996,0.315
instrumentalness,0.000,1.000,0.156


In [10]:
# Canciones con tempo = 0 (suelen ser pistas de ruido, lluvia o silencio)
kaggle.loc[kaggle["tempo"] == 0, ["track_name", "artists", "track_genre", "tempo", "time_signature", "duration_ms"]].head(10)

,track_name,artists,track_genre,tempo,time_signature,duration_ms
4131,The Departure,Max Richter;Lang Lang,ambient,0.0,0,151506
4379,The End of Childhood (feat. Jack Liebeck),Dario Marianelli;Jack Liebeck;Benjamin Wallfisch,ambient,0.0,0,73266
4664,Ferme Les Yeux,Sylvain Chauveau,ambient,0.0,0,68493
45670,Campomoro,Little Symphony,guitar,0.0,0,148711
45720,Ritornello,Little Symphony,guitar,0.0,0,102000
45729,Amager,Little Symphony,guitar,0.0,0,133624
45773,Ionian,Little Symphony,guitar,0.0,0,93125
59221,Yes! No! I Mean Yes! No I Don't!,Leila Bela,iranian,0.0,0,38333
59310,The Exorsism Begins...,Leila Bela,iranian,0.0,0,8586
59473,Leaving Her Quantum Journey Behind...,Leila Bela,iranian,0.0,0,50186


### 2.4 Tipos

In [11]:
esperado = {
    "explicit":    "booleano",
    "key":         "entero",
    "mode":        "entero",
    "track_genre": "texto",
}
def tipo_ok(serie, tipo):
    if tipo == "booleano": return pd.api.types.is_bool_dtype(serie)
    if tipo == "entero":   return pd.api.types.is_integer_dtype(serie)
    if tipo == "texto":    return pd.api.types.is_string_dtype(serie) or pd.api.types.is_object_dtype(serie)

tipos = pd.DataFrame({
    "dtype leído": {c: str(kaggle[c].dtype) for c in esperado},
    "esperado": esperado,
    "correcto": {c: tipo_ok(kaggle[c], t) for c, t in esperado.items()},
})
tipos

,dtype leído,esperado,correcto
explicit,bool,booleano,True
key,int64,entero,True
mode,int64,entero,True
track_genre,object,texto,True


In [12]:
kaggle.dtypes.to_frame("dtype").T

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
dtype,object,object,object,object,int64,int64,bool,float64,float64,int64,float64,int64,float64,float64,float64,float64,float64,float64,int64,object


## 3. Historial de Spotify

Se leen todos los `Streaming_History_Audio_*.json` (o `endsong_*.json`) de cada persona. Cada registro se reduce a `CAMPOS_HISTORIAL` **antes** de crear el DataFrame, así que los campos personales no se guardan en memoria ni aparecen en ninguna salida.

In [13]:
def leer_historial(carpeta: Path, persona: str) -> pd.DataFrame:
    ficheros = sorted(
        set(glob.glob(str(carpeta / "**" / "Streaming_History_Audio_*.json"), recursive=True))
        | set(glob.glob(str(carpeta / "**" / "endsong_*.json"), recursive=True))
    )
    if not ficheros:
        raise FileNotFoundError(f"No hay ficheros de historial en {carpeta}")
    filas = []
    for f in ficheros:
        with open(f, encoding="utf-8") as fh:
            for r in json.load(fh):
                filas.append({k: r.get(k) for k in CAMPOS_HISTORIAL})   # descarta campos personales
    df = pd.DataFrame(filas, columns=CAMPOS_HISTORIAL)
    df["persona"] = persona
    print(f"{persona}: {len(ficheros)} ficheros leídos")
    return df

hist_todo = pd.concat([leer_historial(c, p) for p, c in HISTORIALES.items()], ignore_index=True)

resumen_hist = hist_todo.groupby("persona").agg(
    reproducciones=("ts", "size"),
    con_uri=("spotify_track_uri", lambda s: int(s.notna().sum())),
    desde=("ts", "min"),
    hasta=("ts", "max"),
)
resumen_hist

ander: 20 ficheros leídos
daniel: 10 ficheros leídos


,reproducciones,con_uri,desde,hasta
persona,,,,
ander,263001,262830,2019-05-10T15:41:10Z,2026-10-20T22:00:55Z
daniel,85620,85473,2020-10-02T15:28:05Z,2026-11-17T22:57:49Z


In [14]:
# Comprobación contra las cifras de la sección 8.1 del informe
if "ander" in resumen_hist.index:
    a = resumen_hist.loc["ander"]
    ok = a["reproducciones"] == REFERENCIA_ANDER["reproducciones"] and a["con_uri"] == REFERENCIA_ANDER["con_uri"]
    if ok:
        print("Historial de Ander: coincide con el informe (263.001 / 262.830)")
    else:
        print("AVISO: el historial de Ander no coincide con el informe.",
              f"Leído: {a['reproducciones']} / {a['con_uri']}. Revisar antes de seguir.")

Historial de Ander: coincide con el informe (263.001 / 262.830)


In [15]:
# Solo música: filas con spotify_track_uri
hist = hist_todo[hist_todo["spotify_track_uri"].notna()].copy()
hist["track_id"] = hist["spotify_track_uri"].str.replace("spotify:track:", "", regex=False)
del hist_todo

hist.groupby("persona").agg(
    reproducciones=("track_id", "size"),
    canciones_distintas=("track_id", "nunique"),
)

,reproducciones,canciones_distintas
persona,,
ander,262830,12181
daniel,85473,8362


## 4. Cobertura del cruce con Kaggle

### 4.1 Por `track_id`
- **% de reproducciones** con características de audio: reproducciones cuyo `track_id` está en Kaggle / total de reproducciones de música.
- **% de canciones distintas** con características de audio: `track_id` distintos del historial que están en Kaggle / `track_id` distintos del historial.

In [16]:
ids_kaggle = set(kaggle["track_id"].dropna())
hist["en_kaggle_id"] = hist["track_id"].isin(ids_kaggle)

def cobertura(df, col):
    """% de reproducciones y % de canciones distintas cubiertas, por persona y para el grupo."""
    grupos = [(p, g) for p, g in df.groupby("persona")] + [("grupo", df)]
    out = {}
    for nombre, g in grupos:
        por_cancion = g.groupby("track_id")[col].any()
        out[nombre] = {
            "reproducciones": len(g),
            "reprod. cubiertas": int(g[col].sum()),
            "% reproducciones": round(g[col].mean() * 100, 2),
            "canciones distintas": len(por_cancion),
            "canciones cubiertas": int(por_cancion.sum()),
            "% canciones": round(por_cancion.mean() * 100, 2),
        }
    return pd.DataFrame(out).T

cob_id = cobertura(hist, "en_kaggle_id")
cob_id

,reproducciones,reprod. cubiertas,% reproducciones,canciones distintas,canciones cubiertas,% canciones
ander,262830.0,16362.0,6.23,12181.0,572.0,4.70
daniel,85473.0,5884.0,6.88,8362.0,421.0,5.03
grupo,348303.0,22246.0,6.39,17875.0,734.0,4.11


### 4.2 Por nombre de canción + artista normalizados

Normalización: minúsculas, sin tildes y sin espacios dobles. En Kaggle `artists` puede traer varios artistas separados por `;`, mientras que el historial solo da el artista principal (`master_metadata_album_artist_name`); por eso se genera una clave por cada artista de Kaggle. Solo se intenta este cruce con las canciones que **no** casaron por id.

In [17]:
def normalizar(texto) -> str:
    if not isinstance(texto, str):
        return ""
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))   # quita tildes
    return re.sub(r"\s+", " ", t).strip()                          # quita espacios dobles

claves_kaggle = (
    kaggle[["track_name", "artists"]].dropna()
    .assign(artista=lambda d: d["artists"].str.split(";"))
    .explode("artista")
)
claves_kaggle = set(claves_kaggle["track_name"].map(normalizar) + " || " + claves_kaggle["artista"].map(normalizar))

hist["clave_nombre"] = (hist["master_metadata_track_name"].map(normalizar) + " || "
                        + hist["master_metadata_album_artist_name"].map(normalizar))
hist["en_kaggle_nombre"] = hist["clave_nombre"].isin(claves_kaggle)
hist["en_kaggle_total"] = hist["en_kaggle_id"] | hist["en_kaggle_nombre"]

cob_total = cobertura(hist, "en_kaggle_total")
cob_total

,reproducciones,reprod. cubiertas,% reproducciones,canciones distintas,canciones cubiertas,% canciones
ander,262830.0,20969.0,7.98,12181.0,808.0,6.63
daniel,85473.0,8243.0,9.64,8362.0,631.0,7.55
grupo,348303.0,29212.0,8.39,17875.0,1098.0,6.14


In [18]:
# Cuánto añade el cruce por nombre + artista
aporte = pd.DataFrame({
    "% reprod. por id": cob_id["% reproducciones"],
    "% reprod. id + nombre": cob_total["% reproducciones"],
    "puntos que añade (reprod.)": (cob_total["% reproducciones"] - cob_id["% reproducciones"]).round(2),
    "% canciones por id": cob_id["% canciones"],
    "% canciones id + nombre": cob_total["% canciones"],
    "puntos que añade (canciones)": (cob_total["% canciones"] - cob_id["% canciones"]).round(2),
})
aporte

,% reprod. por id,% reprod. id + nombre,puntos que añade (reprod.),% canciones por id,% canciones id + nombre,puntos que añade (canciones)
ander,6.23,7.98,1.75,4.70,6.63,1.93
daniel,6.88,9.64,2.76,5.03,7.55,2.52
grupo,6.39,8.39,2.00,4.11,6.14,2.03


## 5. Decisión sobre el dataset de respaldo (regla del 40 %)

In [19]:
pct_id = cob_id.loc["grupo", "% reproducciones"]
pct_total = cob_total.loc["grupo", "% reproducciones"]

print(f"Cobertura de reproducciones del grupo por id:           {pct_id:.2f} %")
print(f"Cobertura de reproducciones del grupo por id + nombre:  {pct_total:.2f} %")
print(f"Umbral: {UMBRAL_RESPALDO*100:.0f} %\n")

if pct_total < UMBRAL_RESPALDO * 100:
    decision = ("HACE FALTA el dataset de respaldo (Spotify 1 Million Tracks, amitanshjoshi). "
                "Revisar antes su licencia en Kaggle.")
else:
    decision = "NO hace falta el dataset de respaldo: Kaggle supera el 40 %."
if (pct_id < UMBRAL_RESPALDO * 100) != (pct_total < UMBRAL_RESPALDO * 100):
    decision += " Ojo: por id solo NO llega al 40 %; la decisión depende de aceptar el cruce por nombre."
print("Decisión:", decision)

Cobertura de reproducciones del grupo por id:           6.39 %
Cobertura de reproducciones del grupo por id + nombre:  8.39 %
Umbral: 40 %

Decisión: HACE FALTA el dataset de respaldo (Spotify 1 Million Tracks, amitanshjoshi). Revisar antes su licencia en Kaggle.


## 6. Resumen para el informe (sección 8.2)

Tabla de cifras agregadas, sin datos personales. Se guarda en `data/salida/` (carpeta fuera de Git).

In [20]:
resumen = pd.Series({
    "URL de la ficha": URL_KAGGLE,
    "Fecha de descarga": FECHA_DESCARGA,
    "Filas": len(kaggle_raw),
    "Columnas (con índice sobrante)": kaggle_raw.shape[1],
    "Columna de índice sobrante": ", ".join(columnas_indice) or "no",
    "Tamaño (MB)": round(tam_mb, 1),
    "Celdas nulas": int(nulos["nulos"].sum()),
    "track_id distintos": n_ids,
    "Filas sobrantes por track_id repetido": filas_dup,
    "track_id repetidos con distinto género": ids_varios_generos,
    "Filas con tempo = 0": int(checks["tempo = 0"].sum()),
    "Filas con duration_ms = 0": int(checks["duration_ms = 0"].sum()),
    "popularity fuera de 0-100": int(checks["popularity fuera de 0-100"].sum()),
    "Características fuera de 0-1": int(sum(checks[f"{c} fuera de 0-1"].sum() for c in CARACT_0_1)),
    "Tipos correctos": bool(tipos["correcto"].all()),
    "% reprod. con audio (id)": pct_id,
    "% canciones con audio (id)": cob_id.loc["grupo", "% canciones"],
    "% reprod. con audio (id + nombre)": pct_total,
    "% canciones con audio (id + nombre)": cob_total.loc["grupo", "% canciones"],
    "Decisión respaldo": decision,
}, name="valor")

salida = RAIZ / "data" / "salida"
salida.mkdir(parents=True, exist_ok=True)
resumen.to_csv(salida / "01_resumen_kaggle.csv", header=True)
pd.concat({"por id": cob_id, "id + nombre": cob_total}).to_csv(salida / "01_cobertura_kaggle.csv")
resumen.to_frame()

,valor
URL de la ficha,https://www.kaggle.com/datasets/maharshipandya...
Fecha de descarga,2026-11-19
Filas,114000
Columnas (con índice sobrante),21
Columna de índice sobrante,Unnamed: 0
Tamaño (MB),19.2
Celdas nulas,3
track_id distintos,89741
Filas sobrantes por track_id repetido,24259
track_id repetidos con distinto género,16299


## 7. Dataset de respaldo: Spotify 1 Million Tracks (amitanshjoshi)

Se añade porque la cobertura de Kaggle no llega al 40 %. Misma revisión de calidad, en versión resumida, y cobertura del cruce **combinando** los dos datasets: una canción del historial tiene características de audio si está en cualquiera de ellos.

In [21]:
resp_raw = pd.read_csv(RUTA_RESPALDO)
resp_idx = [c for c in resp_raw.columns if c.startswith("Unnamed")]
resp = resp_raw.drop(columns=resp_idx)

print(f"URL de la ficha: {URL_RESPALDO}")
print(f"Filas: {len(resp_raw):,} | columnas: {resp_raw.shape[1]} (índice sobrante: {resp_idx or 'no'})".replace(",", "."))
print(f"Tamaño en disco: {RUTA_RESPALDO.stat().st_size/1024**2:.1f} MB")
print(f"Años: {resp['year'].min()} - {resp['year'].max()}")
print("Columnas:", list(resp.columns))
resp.head()

URL de la ficha: https://www.kaggle.com/datasets/amitanshjoshi/spotify-1million-tracks
Filas: 1.159.764 | columnas: 20 (índice sobrante: ['Unnamed: 0'])
Tamaño en disco: 167.7 MB
Años: 2000 - 2023
Columnas: ['artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']


,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,Jason Mraz,I Won't Give Up,53QF56cjZA9RTuuMZDrSA6,68,2012,acoustic,0.483,0.303,4,-10.058,1,0.0429,0.6940,0.000000,0.1150,0.139,133.406,240166,3
1,Jason Mraz,93 Million Miles,1s8tP3jP4GZcyHDsjvw218,50,2012,acoustic,0.572,0.454,3,-10.286,1,0.0258,0.4770,0.000014,0.0974,0.515,140.182,216387,4
2,Joshua Hyslop,Do Not Let Me Go,7BRCa8MPiyuvr2VU3O9W0F,57,2012,acoustic,0.409,0.234,3,-13.711,1,0.0323,0.3380,0.000050,0.0895,0.145,139.832,158960,4
3,Boyce Avenue,Fast Car,63wsZUhUZLlh1OsyrZq7sz,58,2012,acoustic,0.392,0.251,10,-9.845,1,0.0363,0.8070,0.000000,0.0797,0.508,204.961,304293,4
4,Andrew Belle,Sky's Still Blue,6nXIYClvJAfi6ujLiKqEq8,54,2012,acoustic,0.430,0.791,6,-5.419,0,0.0302,0.0726,0.019300,0.1100,0.217,171.864,244320,4


In [22]:
r_checks = {
    "celdas nulas":                    int(resp.isna().sum().sum()),
    "filas con algún nulo":            int(resp.isna().any(axis=1).sum()),
    "track_id distintos":              int(resp["track_id"].nunique()),
    "filas sobrantes por track_id":    int(resp.duplicated(subset="track_id").sum()),
    "tempo = 0":                       int((resp["tempo"] == 0).sum()),
    "duration_ms = 0":                 int((resp["duration_ms"] == 0).sum()),
    "popularity fuera de 0-100":       int((~resp["popularity"].between(0, 100)).sum()),
    "time_signature = 0":              int((resp["time_signature"] == 0).sum()),
    "loudness > 0 dB":                 int((resp["loudness"] > 0).sum()),
    "características fuera de 0-1":    int(sum((~resp[c].between(0, 1)).sum() for c in CARACT_0_1)),
}
print("Nulos por columna:", resp.isna().sum()[lambda s: s > 0].to_dict() or "ninguno")
pd.Series(r_checks, name="valor").to_frame()

Nulos por columna: {'artist_name': 15, 'track_name': 1}


,valor
celdas nulas,16
filas con algún nulo,16
track_id distintos,1159764
filas sobrantes por track_id,0
tempo = 0,1199
duration_ms = 0,0
popularity fuera de 0-100,0
time_signature = 0,1228
loudness > 0 dB,1198
características fuera de 0-1,0


In [23]:
# Cruce combinado: Kaggle + respaldo
ids_resp = set(resp["track_id"].dropna())
claves_resp = set(resp["track_name"].map(normalizar) + " || " + resp["artist_name"].map(normalizar))

hist["en_resp_id"] = hist["track_id"].isin(ids_resp)
hist["en_comb_id"] = hist["en_kaggle_id"] | hist["en_resp_id"]
hist["en_comb_total"] = hist["en_comb_id"] | hist["en_kaggle_nombre"] | hist["clave_nombre"].isin(claves_resp)

cob_resp_id = cobertura(hist, "en_resp_id")
cob_comb_id = cobertura(hist, "en_comb_id")
cob_comb_total = cobertura(hist, "en_comb_total")

comparativa = pd.DataFrame({
    "Kaggle (id)": cob_id["% reproducciones"],
    "Kaggle (id + nombre)": cob_total["% reproducciones"],
    "Respaldo solo (id)": cob_resp_id["% reproducciones"],
    "Combinado (id)": cob_comb_id["% reproducciones"],
    "Combinado (id + nombre)": cob_comb_total["% reproducciones"],
})
print("% de REPRODUCCIONES con características de audio")
comparativa

% de REPRODUCCIONES con características de audio


,Kaggle (id),Kaggle (id + nombre),Respaldo solo (id),Combinado (id),Combinado (id + nombre)
ander,6.23,7.98,10.62,13.32,16.51
daniel,6.88,9.64,18.26,21.31,27.79
grupo,6.39,8.39,12.50,15.28,19.28


In [24]:
print("% de CANCIONES DISTINTAS con características de audio")
pd.DataFrame({
    "Kaggle (id)": cob_id["% canciones"],
    "Kaggle (id + nombre)": cob_total["% canciones"],
    "Respaldo solo (id)": cob_resp_id["% canciones"],
    "Combinado (id)": cob_comb_id["% canciones"],
    "Combinado (id + nombre)": cob_comb_total["% canciones"],
})

% de CANCIONES DISTINTAS con características de audio


,Kaggle (id),Kaggle (id + nombre),Respaldo solo (id),Combinado (id),Combinado (id + nombre)
ander,4.70,6.63,10.88,13.06,17.49
daniel,5.03,7.55,11.62,13.93,19.10
grupo,4.11,6.14,10.46,12.32,16.98


In [25]:
# ¿Qué parte de lo no cubierto se explica por música posterior a 2023?
# (el historial no trae año de lanzamiento; se usa el año de la PRIMERA escucha de cada canción como aproximación)
primera = hist.groupby("track_id").agg(primer_anyo=("ts", lambda s: s.min()[:4]),
                                       cubierta=("en_comb_total", "any"),
                                       reprod=("ts", "size"))
tabla = primera.groupby("primer_anyo").agg(canciones=("cubierta", "size"),
                                           pct_canciones_cubiertas=("cubierta", "mean"),
                                           reproducciones=("reprod", "sum"))
tabla["pct_canciones_cubiertas"] = (tabla["pct_canciones_cubiertas"] * 100).round(1)
tabla

,canciones,pct_canciones_cubiertas,reproducciones
primer_anyo,,,
2019,265,50.9,10031
2020,475,41.5,29359
2021,642,31.8,36607
2022,1197,26.0,47456
2023,2312,18.5,71724
2024,3620,15.9,63400
2025,5465,13.4,69214
2026,3899,11.7,20512


In [26]:
pct_comb = cob_comb_total.loc["grupo", "% reproducciones"]
print(f"Cobertura de reproducciones del grupo, Kaggle + respaldo (id + nombre): {pct_comb:.2f} %")
if pct_comb < UMBRAL_RESPALDO * 100:
    decision_final = ("Ni con el respaldo se llega al 40 %. Las características de audio solo estarán para una "
                      "parte de las canciones: el modelo debe poder funcionar sin ellas (la sección 8 decide si se usan).")
else:
    decision_final = "Con el respaldo se supera el 40 %: se usan los dos datasets."
print("Decisión final:", decision_final)

resumen_resp = pd.Series({
    "URL respaldo": URL_RESPALDO,
    "Filas respaldo": len(resp_raw),
    "Columnas respaldo": resp_raw.shape[1],
    "Años respaldo": f"{resp['year'].min()}-{resp['year'].max()}",
    **{f"Respaldo: {k}": v for k, v in r_checks.items()},
    "% reprod. combinado (id)": cob_comb_id.loc["grupo", "% reproducciones"],
    "% canciones combinado (id)": cob_comb_id.loc["grupo", "% canciones"],
    "% reprod. combinado (id + nombre)": pct_comb,
    "% canciones combinado (id + nombre)": cob_comb_total.loc["grupo", "% canciones"],
    "Decisión final": decision_final,
}, name="valor")
pd.concat([resumen, resumen_resp]).to_csv(salida / "01_resumen_kaggle.csv", header=True)
pd.concat({"kaggle id": cob_id, "kaggle id + nombre": cob_total, "respaldo id": cob_resp_id,
           "combinado id": cob_comb_id, "combinado id + nombre": cob_comb_total}).to_csv(salida / "01_cobertura_kaggle.csv")
resumen_resp.to_frame()

Cobertura de reproducciones del grupo, Kaggle + respaldo (id + nombre): 19.28 %
Decisión final: Ni con el respaldo se llega al 40 %. Las características de audio solo estarán para una parte de las canciones: el modelo debe poder funcionar sin ellas (la sección 8 decide si se usan).


,valor
URL respaldo,https://www.kaggle.com/datasets/amitanshjoshi/...
Filas respaldo,1159764
Columnas respaldo,20
Años respaldo,2000-2023
Respaldo: celdas nulas,16
Respaldo: filas con algún nulo,16
Respaldo: track_id distintos,1159764
Respaldo: filas sobrantes por track_id,0
Respaldo: tempo = 0,1199
Respaldo: duration_ms = 0,0


## 8. ¿Aportan las características de audio? (prueba de apoyo)

El motivo principal para quitar los bloques A y D es la **cobertura** (secciones 4 a 7): menos del 20 % de las reproducciones tiene audio, porque los datasets son de 2022-2023 y la mayoría de canciones del grupo se escucharon por primera vez de 2024 en adelante. Esta prueba solo comprueba que, en las canciones que sí tienen audio, tampoco aporta una mejora medible.

- **Filas:** pares persona × canción con al menos 5 escuchas y con audio.
- **Objetivo:** `tasa_no_salto` con la definición de la sección 6 del informe: salto rápido = `reason_end == "fwdbtn"` con `ms_played` < 30.000. Aún sin separar por contexto (franja × tipo de día).
- **Modelo:** regresión Ridge (variables estandarizadas, alpha = 1), un modelo sencillo que no se sobreajusta con unos 700 pares por persona. Validación cruzada de 5 particiones agrupadas por canción (GroupKFold).
- **Comparación, dentro de cada persona:** la media de la persona (modelo base 1), + artista (modelo base 2, gusto por el artista calculado sin la propia canción), + audio y + artista + audio.

`reason_end` y `ms_played` no son datos personales; se leen solo para este cálculo.

In [27]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Se vuelven a leer solo los campos necesarios para el objetivo
def leer_saltos(carpeta, persona):
    campos = ["spotify_track_uri", "reason_end", "ms_played"]
    filas = []
    for f in sorted(glob.glob(str(carpeta / "**" / "Streaming_History_Audio_*.json"), recursive=True)
                    + glob.glob(str(carpeta / "**" / "endsong_*.json"), recursive=True)):
        with open(f, encoding="utf-8") as fh:
            filas += [{k: r.get(k) for k in campos} for r in json.load(fh) if r.get("spotify_track_uri")]
    d = pd.DataFrame(filas)
    d["persona"] = persona
    return d

saltos = pd.concat([leer_saltos(c, p) for p, c in HISTORIALES.items()], ignore_index=True)
saltos["salto_rapido"] = (saltos["reason_end"] == "fwdbtn") & (pd.to_numeric(saltos["ms_played"]) < 30_000)
hist["no_salto"] = (~saltos["salto_rapido"]).to_numpy()

# Hábitos de cada persona: explican por qué sus tasas están en escalas distintas
fw = saltos[saltos["reason_end"] == "fwdbtn"]
habitos = pd.DataFrame({
    "% acaba en trackdone": saltos.groupby("persona")["reason_end"].apply(lambda s: round((s == "trackdone").mean() * 100, 1)),
    "% acaba en fwdbtn": saltos.groupby("persona")["reason_end"].apply(lambda s: round((s == "fwdbtn").mean() * 100, 1)),
    "mediana de ms_played en fwdbtn (s)": (fw.groupby("persona")["ms_played"].median() / 1000).round(1),
    "tasa de no salto por reproducción": hist.groupby("persona")["no_salto"].mean().round(3),
})
habitos

,% acaba en trackdone,% acaba en fwdbtn,mediana de ms_played en fwdbtn (s),tasa de no salto por reproducción
persona,,,,
ander,12.9,60.4,1.1,0.503
daniel,65.2,13.9,21.8,0.927


In [28]:
FEATS = ["danceability", "energy", "key", "loudness", "mode", "speechiness", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo", "time_signature", "duration_ms"]

def clave_kaggle(d):  # primer artista, como en el historial
    return d["track_name"].map(normalizar) + " || " + d["artists"].fillna("").str.split(";").str[0].map(normalizar)

k_id = kaggle.drop_duplicates("track_id").set_index("track_id")[FEATS]
r_id = resp.drop_duplicates("track_id").set_index("track_id")[FEATS]
k_nm = kaggle.assign(c=clave_kaggle).drop_duplicates("c").set_index("c")[FEATS]
r_nm = resp.assign(c=resp["track_name"].map(normalizar) + " || " + resp["artist_name"].map(normalizar)).drop_duplicates("c").set_index("c")[FEATS]
audio_id = pd.concat([k_id, r_id[~r_id.index.isin(k_id.index)]])
audio_nm = pd.concat([k_nm, r_nm[~r_nm.index.isin(k_nm.index)]])

pares = hist.groupby(["persona", "track_id"]).agg(
    n=("no_salto", "size"), tasa=("no_salto", "mean"),
    artista=("master_metadata_album_artist_name", "first"), clave=("clave_nombre", "first")).reset_index()
au = audio_id.reindex(pares["track_id"]).reset_index(drop=True)
au = au.fillna(audio_nm.reindex(pares["clave"]).reset_index(drop=True))
pares[FEATS] = au.to_numpy()
pares["tiene_audio"] = pares["energy"].notna()

p5 = pares[pares["n"] >= 5].copy()
print(f"Pares persona × canción con >= 5 escuchas: {len(p5)} | con audio: {int(p5['tiene_audio'].sum())} ({p5['tiene_audio'].mean()*100:.1f} %)")
print("Tasa media de no salto por canción (pares >= 5):", p5.groupby("persona")["tasa"].mean().round(3).to_dict())

Pares persona × canción con >= 5 escuchas: 6617 | con audio: 1389 (21.0 %)
Tasa media de no salto por canción (pares >= 5): {'ander': 0.46, 'daniel': 0.93}


In [29]:
def validar(df, columnas, con_artista):
    pred = np.zeros(len(df))
    for tr, te in GroupKFold(5).split(df, groups=df["track_id"]):
        dtr, dte = df.iloc[tr], df.iloc[te]
        Xtr = dtr[columnas].copy() if columnas else pd.DataFrame(index=dtr.index)
        Xte = dte[columnas].copy() if columnas else pd.DataFrame(index=dte.index)
        if con_artista:  # gusto por el artista sin contar la propia canción
            suma = dtr.groupby("artista")["tasa"].transform("sum"); cuenta = dtr.groupby("artista")["tasa"].transform("count")
            Xtr["artista"] = np.where(cuenta > 1, (suma - dtr["tasa"]) / (cuenta - 1).clip(lower=1), np.nan)
            Xte["artista"] = dte["artista"].map(dtr.groupby("artista")["tasa"].mean()).to_numpy()
            media = dtr["tasa"].mean()
            Xtr["artista"] = Xtr["artista"].fillna(media); Xte["artista"] = Xte["artista"].fillna(media)
        peso = np.sqrt(dtr["n"])
        if Xtr.shape[1] == 0:   # modelo base 1: media de la persona
            pred[te] = np.average(dtr["tasa"], weights=peso)
            continue
        m = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
        m.fit(Xtr, dtr["tasa"], ridge__sample_weight=peso)
        pred[te] = m.predict(Xte)
    return round(r2_score(df["tasa"], pred), 3), round(mean_absolute_error(df["tasa"], pred), 3)

filas = []
for persona, d in p5[p5["tiene_audio"]].groupby("persona"):
    for nombre, cols, art in [("media de la persona", [], False), ("+ artista", [], True),
                              ("+ audio", FEATS, False), ("+ artista + audio", FEATS, True)]:
        r2, mae = validar(d, cols, art)
        filas.append({"persona": persona, "variables": nombre, "pares": len(d), "R2": r2, "MAE": mae})
prueba_audio = pd.DataFrame(filas)
prueba_audio

,persona,variables,pares,R2,MAE
0,ander,media de la persona,743,-0.009,0.215
1,ander,+ artista,743,0.040,0.210
2,ander,+ audio,743,-0.014,0.216
3,ander,+ artista + audio,743,0.028,0.211
4,daniel,media de la persona,646,-0.003,0.090
5,daniel,+ artista,646,0.090,0.086
6,daniel,+ audio,646,-0.042,0.090
7,daniel,+ artista + audio,646,0.032,0.087


In [30]:
t = prueba_audio.pivot(index="persona", columns="variables", values="R2")
mejora_audio = pd.concat([t["+ audio"] - t["media de la persona"], t["+ artista + audio"] - t["+ artista"]], axis=1).max(axis=1)
print("Mejora de R2 al añadir audio (la mayor de las dos comparaciones):", mejora_audio.round(3).to_dict())
if (mejora_audio <= 0.02).all():
    decision_audio = ("Se QUITAN los bloques A y D. Motivo principal: la cobertura (< 20 % de las reproducciones). "
                      "Apoyo: en las canciones con audio, añadirlo no aporta una mejora medible (R2 +0,02 o menos).")
else:
    decision_audio = ("Por cobertura se quitan del modelo principal, pero el audio mejora el R2 en las canciones que lo tienen: "
                      "valorar un bloque opcional con indicador tiene_audio.")
print(decision_audio)
prueba_audio.to_csv(salida / "01_prueba_audio.csv", index=False)

Mejora de R2 al añadir audio (la mayor de las dos comparaciones): {'ander': -0.005, 'daniel': -0.039}
Se QUITAN los bloques A y D. Motivo principal: la cobertura (< 20 % de las reproducciones). Apoyo: en las canciones con audio, añadirlo no aporta una mejora medible (R2 +0,02 o menos).


## 9. Escala del gusto de cada persona: por qué hay que normalizar antes del mínimo

El recomendador combina el gusto de los miembros con el **mínimo**, y la compatibilidad usa un umbral de 0,6. Si cada persona usa el botón de siguiente de forma muy distinta, sus tasas están en escalas distintas y el mínimo sería casi siempre el de la persona que más salta. Se comprueba con las canciones que los dos han escuchado al menos 5 veces, comparando la tasa bruta con su **percentil** dentro de las tasas de esa persona (0 = la canción que menos aguanta, 1 = la que más).

In [31]:
p5["percentil"] = p5.groupby("persona")["tasa"].rank(pct=True)
ancho = p5.pivot_table(index="track_id", columns="persona", values=["tasa", "percentil"]).dropna()
personas = sorted(p5["persona"].unique())
quien_min_bruto = ancho["tasa"].idxmin(axis=1).value_counts(normalize=True).mul(100).round(1)
quien_min_pct = ancho["percentil"].idxmin(axis=1).value_counts(normalize=True).mul(100).round(1)
escala = pd.DataFrame({
    "% de canciones comunes donde es el mínimo (tasa bruta)": quien_min_bruto,
    "% donde es el mínimo (percentil)": quien_min_pct,
    "% de sus canciones con tasa >= 0,6": p5.groupby("persona")["tasa"].apply(lambda s: round((s >= 0.6).mean() * 100, 1)),
    "% con percentil >= 0,6": p5.groupby("persona")["percentil"].apply(lambda s: round((s >= 0.6).mean() * 100, 1)),
})
print(f"Canciones con >= 5 escuchas de todos los miembros: {len(ancho)}")
escala

Canciones con >= 5 escuchas de todos los miembros: 726


,% de canciones comunes donde es el mínimo (tasa bruta),% donde es el mínimo (percentil),"% de sus canciones con tasa >= 0,6","% con percentil >= 0,6"
ander,96.4,49.0,30.4,39.8
daniel,3.6,51.0,97.5,44.6


In [32]:
dominante = quien_min_bruto.max()
if dominante > 75:
    decision_escala = (f"Con la tasa bruta, una persona es el mínimo en el {dominante:.0f} % de las canciones comunes: "
                       "el gusto de cada miembro se pasa a percentil dentro de sus propias tasas antes de aplicar el mínimo y el umbral de 0,6.")
else:
    decision_escala = "Las escalas son parecidas: el mínimo puede aplicarse sobre la tasa bruta."
print(decision_escala)
escala.to_csv(salida / "01_escala_gusto.csv")

Con la tasa bruta, una persona es el mínimo en el 96 % de las canciones comunes: el gusto de cada miembro se pasa a percentil dentro de sus propias tasas antes de aplicar el mínimo y el umbral de 0,6.
